# 현재 공연장 좌석을 이용한 예정 공급 대리지표 탐색
06번 감사에서 고정 원문 XML의 개별 공연장 ID와 좌석 연결을 확인했다. 추가 결과를 보기 전에 **중심점 500m·시작일정과 현재 좌석의 공동 ID 가용률 90% 이상**으로 표본을 고정한다. 같은 행에서 가용 ID의 예정 시작회차합과 `예정 시작회차×정확한 현재 개별 공연장 좌석`의 합을 비교한다.

2개 노출×2개 결과×10개 업종=40개 계수 검정을 모두 기록한다. 실제 관객·실제 판매좌석·과거 좌석 공급을 새로 측정한 결과가 아니다. 시설 전체 좌석을 곱하지 않는다. 원문에 있는 큰 양의 정수 좌석값을 회귀 결과에 따라 자르거나 제외하지 않는다. 본 연구와 분리된 preliminary 탐색이다.

## 1. 06번 감사와 고정 계획 확인
공식 `(mt10id,mt13id)`가 정확히 연결되고 개별 공연장 좌석이 양의 정수인 후보만 좌석 가용이다. 분모는 12번의 파싱 전 전체 공식 후보 ID다. 06번 입력·원문 해시와 산출물 해시를 재확인한다.

In [ ]:
# 현재 구조 및 과거 고정 명세의 경로를 함께 해석합니다.
from pathlib import Path as _ProjectPath
import json as _bootstrap_json
_project_home = next(p for p in [_ProjectPath.cwd(), *_ProjectPath.cwd().parents]
                     if (p / 'environment/project_paths.ipynb').is_file())
_project_notebook = _bootstrap_json.loads((_project_home / 'environment/project_paths.ipynb').read_text(encoding='utf-8'))
for _project_cell in _project_notebook['cells']:
    if 'project-runtime' in _project_cell.get('metadata', {}).get('tags', []):
        exec(compile(''.join(_project_cell['source']), 'environment/project_paths.ipynb', 'exec'), globals())

from pathlib import Path
import os
for key in ['OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','OMP_NUM_THREADS']:os.environ[key]='1'
import ast,json,hashlib,platform
from datetime import datetime,timezone
import numpy as np
import pandas as pd
from scipy import linalg,stats
from statsmodels.stats.multitest import multipletests
from IPython.display import display,Markdown
ROOT=PROJECT_ROOT
REPORTS=project_path('jupyter-notebook/sandbox/reports');DATA=project_path('jupyter-notebook/sandbox/data')
sha=lambda path:hashlib.sha256(path.read_bytes()).hexdigest()
inputs={}
def register(relative,expected=None):
    path=project_path(relative);digest=sha(path)
    if expected is not None:assert digest==expected,relative
    if relative not in inputs:inputs[relative]={'path':relative,'sha256_before':digest,'expected_sha256':expected}
    return path
s06=project_json_loads(register('jupyter-notebook/sandbox/reports/06_summary.json').read_text(encoding='utf-8'))
assert s06['status']=='complete' and s06['all_inputs_unchanged'] and s06['new_regression_fits']==0
assert s06['seat_quality']['duplicate_exact_hall_key_rows']==0
for record in s06['inputs']:register(record['path'],record['sha256_after'])
for relative,record in s06['outputs'].items():register(relative,record['sha256'])
register('jupyter-notebook/sandbox/notebooks/06_좌석연결과_핵심상권감사.ipynb')
s12=project_json_loads((project_path('reports/20261006/revised_quarterly/12_dataset_summary.json')).read_text(encoding='utf-8'))
PLAN=project_json_loads((project_path(s12['plan_path'])).read_text(encoding='utf-8'))
KEY=['area_id','industry_id','quarter'];INDUSTRY_IDS=[f'CS100{i:03d}' for i in range(1,11)]
dtypes={'area_id':str,'industry_id':str,'quarter':str,'mt20id':str,'mt10id':str,'mt13id':str}
panel=pd.read_csv(register(s12['panel_path'],s12['panel_sha256']),dtype=dtypes,float_precision='round_trip')
pq=pd.read_csv(project_path('jupyter-notebook/sandbox/data/06_performance_quarter_seat_audit.csv'),dtype=dtypes,float_precision='round_trip')
links=pd.read_csv(project_path('dataset/analysis/final/revised_quarterly/facility_area_links.csv'),dtype=dtypes,float_precision='round_trip')
aq06=pd.read_csv(project_path('jupyter-notebook/sandbox/data/06_seat_coverage_area_quarter.csv'),dtype=dtypes,float_precision='round_trip')
INDUSTRIES=dict(panel[['industry_id','industry']].drop_duplicates().sort_values('industry_id').to_numpy())
assert list(INDUSTRIES)==INDUSTRY_IDS

In [ ]:
seat_plan={'created_before_fitting':datetime.now(timezone.utc).isoformat(),
    'exploratory':True,'confirmatory':False,'source_audit_sha256':inputs['jupyter-notebook/sandbox/reports/06_summary.json']['sha256_before'],
    'spatial_definition':'centroid500','joint_ID_availability_threshold_pct':90,
    'joint_ID_numerator':'same source quarter start-eligible AND unique same-facility exact hall ID with positive integer current hall seats',
    'joint_ID_denominator':'all original official candidate performance IDs before schedule or seat filtering',
    'exposures':['known_joint_scheduled_starts','known_current_hall_seat_start_product'],
    'outcomes':['log1p_total_sales_won','log1p_night_sales_21_24_won'],
    'planned_systems':4,'planned_tests':40,'minimum_entity_quarters':2,'minimum_industry_clusters':20,
    'fixed_effects':['area_industry','industry_quarter'],'cluster':'area_id','multiplicity':'BH40 all attempted coefficients, unestimated p=1',
    'facility_total_capacity_used':False,'large_integer_seat_values_capped':False,
    'historical_seat_validity_verified':False,'seat_supply_is_audience':False,
    'same_candidate_rows_for_exposure_comparison':True}
plan_path=DATA/'07_analysis_plan.json'
plan_path.write_text(json.dumps(seat_plan,ensure_ascii=False,indent=2),encoding='utf-8')
seat_plan_sha=sha(plan_path)
assert PLAN['minimum_industry_clusters']==20 and PLAN['minimum_observations_per_area_industry']==2
print('추가 탐색 고정: 4개 시스템·40개 계수, joint ID 가용률90%, 큰값 캡핑 없음')

## 2. 가용 회차합과 개별 좌석 가중합
좌석 미가용 후보의 곱셈 항은 결측으로 두고 가용한 항만 더한 부분 합계를 만든다. 누락 좌석을 0석으로 해석하지 않는다. 후보가 없을 때의 0은 고정 공개등록 범위 안의 0이다.

두 노출은 모두 **같은 joint 가용 ID**에서 계산한다. A를 기존 전체 가용 회차합으로 사용하면 서로 다른 공연 구성까지 비교하게 되므로 그렇게 하지 않는다.

In [ ]:
valid=pq.joint_start_and_seat_eligible
assert valid.equals(pq.eligible_id_quarter & pq.current_hall_seat_available)
assert pq.loc[valid,'hall_seats'].gt(0).all() and np.equal(pq.loc[valid,'hall_seats']%1,0).all()
pq['joint_start_term']=pq.known_scheduled_start_count.where(valid)
pq['current_hall_seat_start_term']=(pq.known_scheduled_start_count*pq.hall_seats).where(valid)
assert pq.loc[~valid,['joint_start_term','current_hall_seat_start_term']].isna().all().all()
rel=links.loc[links.spatial_definition.eq('centroid500'),['area_id','mt10id']].merge(pq,on='mt10id',validate='many_to_many')
assert not rel.duplicated(['area_id','quarter','mt20id']).any()
agg=rel.groupby(['area_id','quarter']).agg(candidate_id_count=('mt20id','nunique'),
    joint_eligible_id_count=('joint_start_and_seat_eligible','sum'),
    known_joint_scheduled_starts=('joint_start_term','sum'),
    known_current_hall_seat_start_product=('current_hall_seat_start_term','sum')).reset_index()
base=panel[['area_id','quarter']].drop_duplicates()
aq=base.merge(agg,on=['area_id','quarter'],how='left',validate='one_to_one')
for column in ['candidate_id_count','joint_eligible_id_count','known_joint_scheduled_starts','known_current_hall_seat_start_product']:
    aq[column]=aq[column].fillna(0).astype('int64')
aq['joint_availability_pct']=100*aq.joint_eligible_id_count/aq.candidate_id_count.replace(0,np.nan)
aq.loc[aq.candidate_id_count.eq(0),'joint_availability_pct']=100.0
aq['zero_registered_candidate']=aq.candidate_id_count.eq(0)
check=aq.merge(aq06.loc[aq06.spatial_definition.eq('centroid500'),['area_id','quarter','joint_availability_pct']],on=['area_id','quarter'],validate='one_to_one',suffixes=('_07','_06'))
assert np.allclose(check.joint_availability_pct_07,check.joint_availability_pct_06,rtol=0,atol=1e-10)
assert len(aq)==1812 and aq.joint_eligible_id_count.le(aq.candidate_id_count).all()
assert aq.loc[aq.zero_registered_candidate,['known_joint_scheduled_starts','known_current_hall_seat_start_product']].eq(0).all().all()
display(aq[['candidate_id_count','joint_eligible_id_count','joint_availability_pct','known_joint_scheduled_starts','known_current_hall_seat_start_product']].describe())

## 3. 공통 표본과 핵심 상권 포함 여부
총매출 양수·야간매출 비결측 0 이상을 두 결과에 공통 적용한다. joint 가용률 90% 및 상권×업종의 2분기 관측을 요구한다. 실제 제공된 야간 0은 유지한다. 좌석 조건은 단순 시작일정 90%보다 표본을 더 좁힐 수 있다.

In [ ]:
merged=panel.merge(aq,on=['area_id','quarter'],validate='many_to_one')
common=merged.total_sales.gt(0)&merged.night_sales.notna()&merged.night_sales.ge(0)
observed=merged.loc[common & merged.joint_availability_pct.ge(90)].copy()
counts=observed.groupby(['area_id','industry_id']).quarter.transform('nunique')
frame=observed.loc[counts.ge(2)].sort_values(KEY).reset_index(drop=True)
frame['log1p_total_sales_won']=np.log1p(frame.total_sales)
frame['log1p_night_sales_21_24_won']=np.log1p(frame.night_sales)
expected=next(row for row in s06['seat_sample_availability'] if row['spatial_definition']=='centroid500' and row['joint_threshold_pct']==90)
assert len(frame)==expected['repeated_entity_rows'] and frame.area_id.nunique()==expected['repeated_entity_areas']
core_rows=[]
for area_id in ['3120012','3120103']:
    full=merged.loc[common & merged.area_id.eq(area_id)]
    kept=frame.loc[frame.area_id.eq(area_id)]
    core_rows.append(dict(area_id=area_id,area_name=full.area_name.iloc[0],provided_rows=len(full),
        joint90_repeated_rows=len(kept),joint90_quarters=int(kept.quarter.nunique()),
        retained_pct=100*len(kept)/len(full)))
core=pd.DataFrame(core_rows)
display(pd.DataFrame([{'common_rows_after_joint90':len(observed),'repeated_rows':len(frame),
    'repeated_areas':frame.area_id.nunique(),'night_zero_rows':int(frame.night_sales.eq(0).sum())}]))
display(core)

## 4. 검증된 고정효과·군집 함수 재사용
13번의 FWL와 결합 상권군집 CR1 함수를 함수 정의만 읽어 재사용한다. 업종별 기울기 시스템의 전체 상권 수 G−1을 t 자유도로 쓰며, 업종별 관측 상권 수도 따로 기록한다. 결과에 따라 표본·좌석 상한·가용률을 바꾸지 않는다.

In [ ]:
def import_functions(relative,names):
    notebook=project_json_loads((project_path(relative)).read_text(encoding='utf-8'));nodes=[]
    for cell in notebook['cells']:
        if cell['cell_type']!='code':continue
        source=''.join(cell['source'])
        if any(line.lstrip().startswith(('%','!')) for line in source.splitlines()):continue
        nodes.extend(node for node in ast.parse(source).body if isinstance(node,ast.FunctionDef) and node.name in names)
    assert sorted(n.name for n in nodes)==sorted(names)
    exec(compile(ast.fix_missing_locations(ast.Module(body=nodes,type_ignores=[])),relative,'exec'),globals())
    return {n.name:hashlib.sha256(ast.dump(n).encode()).hexdigest() for n in nodes}
function_hashes={**import_functions('jupyter-notebook/20261006/notebooks/05_외식업_고정효과분석.ipynb',['absorb_effects']),
    **import_functions('jupyter-notebook/20261006/notebooks/13_분기매출액_수정분석.ipynb',['key_hash','fit_system','estimate_row'])}
SCENARIOS={'joint_start_count':'known_joint_scheduled_starts','current_hall_seat_weighted_starts':'known_current_hall_seat_start_product'}
OUTCOMES=['log1p_total_sales_won','log1p_night_sales_21_24_won']
results=[];sample_rows=[];member=[]
for scenario,exposure_column in SCENARIOS.items():
    f=frame.copy();f['x']=np.log1p(f[exposure_column])
    for outcome in OUTCOMES:
        failure=None
        try:m=fit_system(f,outcome)
        except Exception as exc:
            failure=f'{type(exc).__name__}: {str(exc)[:500]}'
            m={'status':'model_failed','blocks':[],'eligibility':[],'nobs':0,'clusters':0,'rank':0}
        estimated=pd.concat([b['frame'][KEY] for b in m['blocks']],ignore_index=True) if m['blocks'] else f.iloc[:0]
        sample_rows.append(dict(scenario=scenario,exposure=exposure_column,outcome=outcome,status=m['status'],failure_reason=failure,
            nobs_common=len(f),nobs_model=m['nobs'],clusters_model=m['clusters'],inference_df=m.get('df',np.nan),
            estimated_industries=len(m['blocks']),sample_key_sha256=key_hash(f),estimated_sample_key_sha256=key_hash(estimated)))
        positions={b['industry_id']:j for j,b in enumerate(m['blocks'])}
        for b in m['blocks']:member.append(b['frame'][KEY].assign(scenario=scenario,outcome=outcome))
        for ind in INDUSTRY_IDS:
            part=f.loc[f.industry_id.eq(ind)]
            e=next((row for row in m['eligibility'] if row['industry_id']==ind),{})
            row=dict(scenario=scenario,exposure=exposure_column,outcome=outcome,industry_id=ind,industry=INDUSTRIES[ind],
                status=e.get('status','model_failed' if failure else 'not_estimable'),failure_reason=failure,
                nobs=len(part),clusters=int(part.area_id.nunique()),nobs_industry=len(part),clusters_industry=int(part.area_id.nunique()),
                nobs_total=m['nobs'],clusters_total=m['clusters'],inference_df=m.get('df',np.nan),
                design_rank=m['rank'],cluster_correction=m.get('correction',np.nan),
                coefficient=np.nan,standard_error=np.nan,ci_low=np.nan,ci_high=np.nan,p_value=np.nan)
            if ind in positions:row.update(estimate_row(m,positions[ind]))
            results.append(row)
results=pd.DataFrame(results);samples=pd.DataFrame(sample_rows)
assert len(results)==40 and results.groupby(['scenario','outcome']).size().eq(10).all()
assert samples.sample_key_sha256.nunique()==1
same_estimated_samples=bool(samples.estimated_sample_key_sha256.nunique()==1)
display(samples)

## 5. 全40개 검정과 대비의 단위
BH40은 추가 탐색 전체 40개를 한 군으로 보정한다. 미추정·실패는 계산에서 p=1로 유지한다. 결과별 BH10과 사양별 BH20도 함께 저장하지만 더 작은 검정군만 골라 결론내리지 않는다. root의 전체 실제자료 탐색 장부에도 이 40개를 추가한다.

대비는 `1+노출` 2배에 대한 `1+원화매출`의 조건부 기하평균 상대차이다. 회차합 2배와 좌석 가중합 2배는 다른 변화이므로 계수 차이가 좌석 가중의 측정 개선량이나 관객 효과를 뜻하지 않는다.

In [ ]:
results['p_for_multiplicity']=results.p_value.where(results.status.eq('estimated'),1).fillna(1)
results['p_bh40']=multipletests(results.p_for_multiplicity,method='fdr_bh')[1]
results['p_bh10']=np.nan;results['p_bh20']=np.nan
for _,indices in results.groupby(['scenario','outcome']).groups.items():results.loc[indices,'p_bh10']=multipletests(results.loc[indices,'p_for_multiplicity'],method='fdr_bh')[1]
for _,indices in results.groupby('scenario').groups.items():results.loc[indices,'p_bh20']=multipletests(results.loc[indices,'p_for_multiplicity'],method='fdr_bh')[1]
for source,target in [('coefficient','doubling_pct'),('ci_low','doubling_ci_low'),('ci_high','doubling_ci_high')]:results[target]=100*np.expm1(results[source]*np.log(2))
results['doubling_definition']='1+selected exposure doubles; relative conditional geometric mean difference of1+sales won'
results['exploratory_not_confirmatory']=True
counts=results.groupby(['scenario','outcome']).agg(attempts=('industry_id','size'),estimated=('status',lambda x:x.eq('estimated').sum()),
    raw_p_below05=('p_value',lambda x:x.lt(.05).sum()),BH40_below05=('p_bh40',lambda x:x.lt(.05).sum()),minimum_p=('p_value','min')).reset_index()
display(counts)
display(results[['scenario','outcome','industry','nobs','clusters','doubling_pct','doubling_ci_low','doubling_ci_high','p_value','p_bh40']])

**해석:** 양의 좌석값은 정수 구조와 공식 ID가 확인됐다는 뜻이다. 실제 이용 가능 좌석, 과거 좌석 변경, 당일 판매량, 실제 관객은 확인되지 않았다. 현재 좌석 조건은 표본을 추가로 선택하므로, 이전 일정만의 90% 결과와 차이가 나더라도 그 차이를 좌석 가중 하나의 효과로 해석하지 않는다.

## 6. 저장과 원본 불변 확인
같은 표본의 4개 시스템·40개 시도를 전수 기록한다. 기존 jupyter-notebook/sandbox/01 결과나 final 결과를 바꾸지 않는다.

In [ ]:
sample_columns=KEY+['area_name','district','total_sales','night_sales','candidate_id_count','joint_eligible_id_count',
    'joint_availability_pct','known_joint_scheduled_starts','known_current_hall_seat_start_product']
data_outputs={'07_area_quarter_exposure.csv':aq,'07_analysis_sample.csv':frame[sample_columns]}
report_outputs={'07_results.csv':results,'07_model_samples.csv':samples,'07_multiplicity_counts.csv':counts,
    '07_core_area_coverage.csv':core,'07_sample_membership.csv':pd.concat(member,ignore_index=True) if member else pd.DataFrame(columns=KEY+['scenario','outcome'])}
output_records={}
for directory,mapping in [(DATA,data_outputs),(REPORTS,report_outputs)]:
    for name,data in mapping.items():
        path=directory/name;data.to_csv(path,index=False,encoding='utf-8-sig')
        output_records[path.relative_to(ROOT).as_posix()]={'rows':len(data),'columns':data.columns.tolist(),'sha256':sha(path)}
assert sha(plan_path)==seat_plan_sha
for relative,record in inputs.items():
    record['sha256_after']=sha(project_path(relative));record['unchanged']=record['sha256_before']==record['sha256_after']
assert all(record['unchanged'] for record in inputs.values())
def finite(value):
    if isinstance(value,dict):return {k:finite(v) for k,v in value.items()}
    if isinstance(value,list):return [finite(v) for v in value]
    if isinstance(value,(float,np.floating)) and not np.isfinite(value):return None
    if isinstance(value,np.generic):return value.item()
    return value
minimum=results.loc[results.p_value.idxmin()].to_dict() if results.p_value.notna().any() else None
summary=finite({'status':'complete','created_utc':datetime.now(timezone.utc).isoformat(),'role':'current_hall_seat_proxy_exploration',
    'exploratory':True,'confirmatory':False,'planned_systems':4,'planned_tests':40,'attempted_tests':len(results),
    'estimated_coefficients':int(results.status.eq('estimated').sum()),'failed_systems':int(samples.status.eq('model_failed').sum()),
    'raw_p_below05':int(results.p_value.lt(.05).sum()),'BH40_below05':int(results.p_bh40.lt(.05).sum()),
    'minimum_raw_p_row':minimum,'plan':seat_plan,'plan_sha256':seat_plan_sha,'plan_path':plan_path.relative_to(ROOT).as_posix(),
    'all_inputs_unchanged':True,'network_requests':0,'historical_seats_verified':False,'actual_audience_observed':False,
    'facility_total_capacity_used':False,'seat_imputation_used':False,'large_seat_values_capped':False,
    'same_candidate_sample_all_systems':True,'same_estimated_sample_all_systems':same_estimated_samples,
    'common_repeated_rows':len(frame),'common_repeated_areas':int(frame.area_id.nunique()),
    'night_zero_rows_preserved':int(frame.night_sales.eq(0).sum()),
    'known_aggregate_rule':'sum only jointly eligible ID terms; omitted seats not zero; both exposure sums use identical IDs',
    'counts':counts.to_dict('records'),'core_area_coverage':core.to_dict('records'),'model_samples':samples.to_dict('records'),
    'function_AST_sha256':function_hashes,'inputs':list(inputs.values()),'outputs':output_records,
    'limitations':['current hall capacity is not historical or realized ticket supply and not audience',
        'joint90% ID availability retains partial scheduled sums and is not90% seat or audience coverage',
        'seat availability changes the analysis sample relative to schedule-only90%',
        'all40 attempts join the broader exploratory search; BH40 alone does not make a confirmatory finding']})
(REPORTS/'07_summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
print(f'완료: 4시스템·40검정, 같은 후보표본 {len(frame):,}행·{frame.area_id.nunique()}상권; 입력 {len(inputs)}개 불변')